To create text embeddings using Keras, you must first convert raw text into sequences of integers using the TextVectorization layer and then map those integers to dense vectors using the Embedding layer

#1. Vectorize the Text

Vectorization layer automatically standardizes text (lowercasing and removing punctuation), splits strings into tokens, and maps each unique token to a specific integer index.

In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization, Embedding

# Example raw text corpus
data = [
    "The cat sat on the mat.",
    "The dog barked at the mailman.",
    "Deep learning and natural language processing are fascinating."
]

In [3]:
# Initialize the vectorization layer
max_features = 10000  # Maximum vocabulary size

sequence_length = 10  # Pad or truncate every sentence to 10 tokens

vectorizer = TextVectorization(
    max_tokens=max_features,
    output_sequence_length=sequence_length
)

In [4]:
# Fit the layer to your text data to build the vocabulary
vectorizer.adapt(data)

In [5]:
# Transform text to integer sequences
integer_sequences = vectorizer(data)

print("Integer Sequences:\n", integer_sequences.numpy())

Integer Sequences:
 [[ 2 14  3  5  2  7  0  0  0  0]
 [ 2 12 15 16  2  8  0  0  0  0]
 [13  9 18  6 10  4 17 11  0  0]]


#2. Add the Keras Embedding Layer

The Embedding layer serves as a trainable lookup table. It translates each token's integer ID into a fixed-size dense vector of real numbers

In [6]:
# Configure dimensions
vocabulary_size = len(vectorizer.get_vocabulary())

embedding_dim = 16  # Dimension of the dense vector

# Initialize the Embedding layer
embedding_layer = Embedding(
    input_dim=vocabulary_size,
    output_dim=embedding_dim
)

In [7]:
# Convert integer sequences to continuous text embeddings
text_embeddings = embedding_layer(integer_sequences)
text_embeddings

<tf.Tensor: shape=(3, 10, 16), dtype=float32, numpy=
array([[[-4.03231271e-02, -3.63338217e-02,  2.80031674e-02,
          1.42881759e-02, -1.89230591e-03, -2.08027363e-02,
          1.46462433e-02, -3.78971212e-02,  4.11273874e-02,
          2.46143006e-02,  8.96617025e-03,  1.01135485e-02,
         -2.03889962e-02, -4.47779894e-02,  4.48888279e-02,
         -1.06882565e-02],
        [-5.17769903e-03,  2.79477574e-02, -3.49686295e-03,
         -3.21196318e-02,  3.23803537e-02, -1.01091489e-02,
         -2.38597393e-03,  5.81815094e-03,  1.27199031e-02,
         -4.08381112e-02, -3.09986994e-03, -2.59237364e-03,
          1.95704773e-03,  2.76621617e-02, -2.59105098e-02,
         -1.16030201e-02],
        [-3.87613066e-02,  4.48035039e-02, -1.80363767e-02,
          8.79240036e-03, -9.64418799e-03,  4.31258120e-02,
         -3.28187495e-02,  2.77794264e-02,  6.88564032e-04,
          4.53757979e-02, -1.45526975e-03,  1.16102695e-02,
         -4.11259905e-02,  2.11630799e-02, -3.2418705

In [8]:
print("\nEmbedding Tensor Shape (Batch, Sequence, Vector):")
print(text_embeddings.shape)  # Output: (3, 10, 16)


Embedding Tensor Shape (Batch, Sequence, Vector):
(3, 10, 16)


#3. Integrate into a Trainable Model
In deep learning architectures, you typically stack these layers consecutively inside a Keras Sequential Model so the network can optimize the embedding weights during training:

In [9]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GlobalAveragePooling1D, Dense

model = Sequential([
    # Input layer accepts raw strings directly
    tf.keras.Input(shape=(1,), dtype=tf.string),

    # 1. Step: Tokenize & Index
    vectorizer,

    # 2. Step: Map to 16-dimensional continuous vectors
    embedding_layer,

    # 3. Step: Pool sequence dimensions to get a static document representation
    GlobalAveragePooling1D(),

    # 4. Step: Classification or downstream task
    Dense(1, activation='sigmoid')
])

# View structure
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ text_vectorization_1            │ (None, 10)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 10, 16)         │           304 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 321 (1.25 KB)

 Trainable params: 321 (1.25 KB)

 Non-trainable params: 0 (0.00 B)